In [ ]:
from Datasets.datasets import Dataset
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
os.environ['NationalityDatasets'] = "./data/"

dt = Dataset()
data = dt.get_data()
train = dt.get_train()
test = dt.get_test()
validation = dt.get_validation()
train_validation = dt.get_train_validation()


In [ ]:
data

In [ ]:
model_dir = "./data/"

In [ ]:
from torch.utils.data import DataLoader
import torch
from Models.Transformer.ByT5 import model_init, MyDataset, DataCollator
from transformers import AutoTokenizer

checkpoint = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = model_init(checkpoint)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
train_ds = MyDataset(train, 'level_3')
val_ds = MyDataset(validation, 'level_3')
data_collator = DataCollator(tokenizer)
train_loader = DataLoader(train_ds, batch_size=128, collate_fn=data_collator)
val_loader = DataLoader(val_ds, batch_size=128, collate_fn=data_collator)

In [ ]:
@torch.no_grad()
def eval_model(model, val_loader, tokenizer):
    model.eval()
    total = 0
    count = 0
    stream = tqdm(val_loader)
    for batch in stream:
        batch = dict_to_cuda(batch)
        loss = model(**batch).loss
        total += loss.item()
        count += 1
        stream.set_description_str(desc=f"val_edit_dist {total / count:.3f}")

    model.train()
    return total / count
def dict_to_cuda(batch):
    return {k: v.cuda() for k, v in batch.items()}

In [ ]:
from datasets import tqdm

max_epochs = 10
best = float("inf")
trigger = 0

for epoch in range(max_epochs):
    stream = tqdm(train_loader)
    total = 0
    count = 0
    for batch in stream:
        batch = dict_to_cuda(batch)
        loss = model(**batch).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        count += 1
        stream.set_description_str(
            f"Epoch {epoch}/{max_epochs} | Loss {total / count : .3f}"
        )
    loss = total / count
    eval_metric = eval_model(model, val_loader, tokenizer)

    print(
        f"Epoch {epoch}/{max_epochs} | Loss {loss: .3f} | Val Loss {eval_metric: .4f}"
    )

    trigger += 1

    if eval_metric < best:
        to_save = {"model": model.state_dict(), "optimizer": optimizer}
        best = eval_metric
        trigger = 0
        torch.save(to_save, model_dir)

    if trigger > 5:
        break

In [ ]:
torch.cuda.empty_cache() 

In [ ]:
print('hi')

In [ ]:
from Models.Transformer.ByT5 import Pipeline

pipeline = Pipeline(model_dir, device="cuda")

In [ ]:
pipeline.generate("Faeze Ghorbanpour")

In [ ]:
outputs = []
stream = tqdm(test["name"])
for text in stream:
    output = pipeline.generate(text)
    stream.set_description_str(output)
    outputs.append(output)

In [ ]:

test["expected"] = outputs
test.to_csv("level_3_predict.csv", index=False)